# Pipeline de preprocesamiento EEG

BIDS crudo `.set`/`.fdt` (61 canales, 500 Hz) -> **limpieza** (detección +
interpolación de canales malos, pasa-banda 1-40 Hz, notch 50 Hz) ->
**segmentación** (épocas de 4 s sin traslape, tope de 75, según Cui et al.
2026) -> **extracción de features** (PSD de Welch por época/electrodo,
potencia relativa theta = theta / [1,40] Hz) -> **agregación**:

- `eeg_theta_per_electrode.csv`: media + std (variabilidad) entre épocas,
  por sujeto/condición/task/electrodo -- alimenta la **vista de topomap**
  (Q1, Q2).
- `eeg_theta_epochs_roi.csv`: potencia theta por época promediada dentro de
  cada ROI (frontal / centro-temporal / otro) -- alimenta la **vista de
  dinámica intra-grabación** (Q6).

Por defecto busca el dataset completo en `../../../ds004902` (raíz del
repo, ver `acquisition.md`) y si no está, usa la muestra de la Semana 4.
Poner `LIMIT` (abajo) en un número chico (ej. 4) para iterar rápido; volver
a `None` para la corrida completa de 218 grabaciones (~15-20 min).

In [1]:
from __future__ import annotations

import re
import warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("ERROR")

REPO_ROOT = Path.cwd().resolve().parents[2]  # deliveries/week06/code -> raíz del repo
FULL_BIDS_ROOT = REPO_ROOT / "ds004902"
SAMPLE_BIDS_ROOT = REPO_ROOT / "deliveries" / "week04" / "data" / "sample"
PARTICIPANTS_TSV = REPO_ROOT / "deliveries" / "week04" / "data" / "participants.tsv"
OUT_DIR = REPO_ROOT / "deliveries" / "week06" / "data" / "processed"

EPOCH_DURATION_S = 4.0
MAX_EPOCHS = 75
BAND_FULL = (1.0, 40.0)
BAND_THETA = (4.0, 8.0)
BAD_CHANNEL_Z = 3.0

BIDS_ROOT = FULL_BIDS_ROOT if FULL_BIDS_ROOT.exists() else SAMPLE_BIDS_ROOT
LIMIT = None  # poner un entero chico (ej. 4) para una corrida de prueba rápida
print(f"Usando BIDS_ROOT = {BIDS_ROOT}")

Usando BIDS_ROOT = C:\Users\jimen\Downloads\data-visualization-project\ds004902


In [2]:
# Membresía de ROI verificada contra los 61 nombres de canal presentes en
# ds004902 (montaje 10-20 extendido). Los conteos coinciden con Cui et al.
# (2026): 16 frontal, 28 centro-temporal. Todo lo demás (parietal/occipital,
# 17 canales) se agrupa como "other".
FRONTAL_ROI = [
    "Fp1", "Fp2", "Fpz", "AF3", "AF4", "AF7", "AF8",
    "F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8", "Fz",
]
CENTROTEMPORAL_ROI = [
    "FC1", "FC2", "FC3", "FC4", "FC5", "FC6", "FT7", "FT8",
    "C1", "C2", "C3", "C4", "C5", "C6", "Cz", "T7", "T8",
    "CP1", "CP2", "CP3", "CP4", "CP5", "CP6", "CPz", "TP7", "TP8", "TP9", "TP10",
]


def roi_for_channel(ch_name: str) -> str:
    if ch_name in FRONTAL_ROI:
        return "frontal"
    if ch_name in CENTROTEMPORAL_ROI:
        return "centro_temporal"
    return "other"

In [3]:
FNAME_RE = re.compile(r"(sub-\d+)_(ses-\d+)_task-(eyesopen|eyesclosed)_eeg\.set$")


def find_recordings(bids_root: Path) -> list[dict]:
    recordings = []
    for set_path in sorted(bids_root.rglob("*_eeg.set")):
        m = FNAME_RE.search(set_path.name)
        if not m:
            continue
        participant_id, session, task = m.groups()
        electrodes_tsv = set_path.parent / f"{participant_id}_{session}_electrodes.tsv"
        recordings.append(
            {
                "participant_id": participant_id,
                "session": session,
                "task": task,
                "set_path": set_path,
                "electrodes_tsv": electrodes_tsv,
            }
        )
    return recordings


def session_to_condition(participants_df: pd.DataFrame) -> dict[tuple[str, str], str]:
    """Mapea (participant_id, session) -> 'NS' o 'SD' usando SessionOrder."""
    mapping = {}
    for _, row in participants_df.iterrows():
        pid = row["participant_id"]
        first, second = row["SessionOrder"].split("->")
        mapping[(pid, "ses-1")] = first
        mapping[(pid, "ses-2")] = second
    return mapping

In [4]:
def build_montage(electrodes_tsv: Path):
    if not electrodes_tsv.exists():
        return None
    df = pd.read_csv(electrodes_tsv, sep="\t")
    # Las coordenadas están en milímetros relativas al centro de la cabeza; MNE espera metros.
    ch_pos = {row["name"]: np.array([row["x"], row["y"], row["z"]]) / 1000.0 for _, row in df.iterrows()}
    return mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame="head")


def detect_bad_channels(raw: mne.io.BaseRaw) -> list[str]:
    data = raw.get_data()
    log_var = np.log(np.var(data, axis=1) + 1e-20)
    z = (log_var - log_var.mean()) / log_var.std()
    return [ch for ch, zi in zip(raw.ch_names, z) if abs(zi) > BAD_CHANNEL_Z]

In [5]:
def process_recording(rec: dict):
    raw = mne.io.read_raw_eeglab(rec["set_path"], preload=True, verbose=False)

    montage = build_montage(rec["electrodes_tsv"])
    if montage is not None:
        raw.set_montage(montage, on_missing="ignore", verbose=False)

    raw.notch_filter(50.0, verbose=False)
    raw.filter(*BAND_FULL, verbose=False)

    bads = detect_bad_channels(raw)
    n_bad = len(bads)
    if bads and montage is not None and n_bad < 0.2 * len(raw.ch_names):
        raw.info["bads"] = bads
        raw.interpolate_bads(reset_bads=True, verbose=False)

    epochs = mne.make_fixed_length_epochs(raw, duration=EPOCH_DURATION_S, preload=True, verbose=False)
    n_epochs_available = len(epochs)
    epochs = epochs[: min(MAX_EPOCHS, n_epochs_available)]
    n_epochs = len(epochs)
    if n_epochs == 0:
        return None

    spectrum = epochs.compute_psd(method="welch", fmin=BAND_FULL[0], fmax=BAND_FULL[1], verbose=False)
    psds, freqs = spectrum.get_data(return_freqs=True)  # (n_epochs, n_channels, n_freqs)

    theta_mask = (freqs >= BAND_THETA[0]) & (freqs <= BAND_THETA[1])
    theta_power = psds[:, :, theta_mask].sum(axis=2)  # (n_epochs, n_channels)
    total_power = psds.sum(axis=2)  # (n_epochs, n_channels)
    relative_theta = theta_power / total_power  # (n_epochs, n_channels)

    ch_names = epochs.ch_names
    per_electrode = pd.DataFrame(
        {
            "electrode": ch_names,
            "theta_mean": relative_theta.mean(axis=0),
            "theta_std": relative_theta.std(axis=0),
            "interpolated": [ch in bads for ch in ch_names],
        }
    )

    roi_of = np.array([roi_for_channel(ch) for ch in ch_names])
    roi_rows = []
    for roi in ("frontal", "centro_temporal", "other"):
        cols = roi_of == roi
        if not cols.any():
            continue
        roi_series = relative_theta[:, cols].mean(axis=1)  # (n_epochs,)
        for epoch_idx, value in enumerate(roi_series):
            roi_rows.append({"epoch_idx": epoch_idx, "roi": roi, "theta_power": value})
    per_epoch_roi = pd.DataFrame(roi_rows)

    meta = {"n_epochs_available": n_epochs_available, "n_epochs_used": n_epochs, "n_bad_channels": n_bad}
    return per_electrode, per_epoch_roi, meta

In [6]:
participants_df = pd.read_csv(PARTICIPANTS_TSV, sep="\t")
cond_map = session_to_condition(participants_df)

recordings = find_recordings(BIDS_ROOT)
if LIMIT:
    recordings = recordings[:LIMIT]
print(f"Se encontraron {len(recordings)} grabaciones en {BIDS_ROOT}")

Se encontraron 218 grabaciones en C:\Users\jimen\Downloads\data-visualization-project\ds004902


In [7]:
electrode_rows = []
roi_rows = []
skipped = []

for i, rec in enumerate(recordings, 1):
    pid, ses, task = rec["participant_id"], rec["session"], rec["task"]
    condition = cond_map.get((pid, ses))
    if condition is None:
        skipped.append((pid, ses, task, "no SessionOrder mapping"))
        continue
    try:
        result = process_recording(rec)
    except Exception as exc:  # noqa: BLE001 - registrar y seguir a través de las 218 grabaciones
        skipped.append((pid, ses, task, str(exc)))
        continue
    if result is None:
        skipped.append((pid, ses, task, "zero usable epochs"))
        continue

    per_electrode, per_epoch_roi, meta = result
    per_electrode["participant_id"] = pid
    per_electrode["condition"] = condition
    per_electrode["task"] = task
    per_electrode["n_epochs_used"] = meta["n_epochs_used"]
    per_electrode["n_bad_channels"] = meta["n_bad_channels"]
    electrode_rows.append(per_electrode)

    per_epoch_roi["participant_id"] = pid
    per_epoch_roi["condition"] = condition
    per_epoch_roi["task"] = task
    roi_rows.append(per_epoch_roi)

    print(f"[{i}/{len(recordings)}] {pid} {ses}({condition}) {task}: "
          f"{meta['n_epochs_used']} épocas, {meta['n_bad_channels']} canales malos")

print("\nListo.")

[1/218] sub-01 ses-1(NS) eyesclosed: 75 épocas, 0 canales malos


[2/218] sub-01 ses-1(NS) eyesopen: 75 épocas, 1 canales malos


[3/218] sub-01 ses-2(SD) eyesclosed: 75 épocas, 0 canales malos


[5/218] sub-02 ses-1(NS) eyesclosed: 75 épocas, 0 canales malos


[6/218] sub-02 ses-1(NS) eyesopen: 75 épocas, 1 canales malos


...

[214/218] sub-69 ses-2(SD) eyesopen: 75 épocas, 0 canales malos


[215/218] sub-70 ses-1(NS) eyesopen: 75 épocas, 0 canales malos


[216/218] sub-70 ses-2(SD) eyesopen: 75 épocas, 1 canales malos


[217/218] sub-71 ses-1(NS) eyesopen: 75 épocas, 1 canales malos


[218/218] sub-71 ses-2(SD) eyesopen: 75 épocas, 0 canales malos

Listo.


In [8]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

if electrode_rows:
    electrode_df = pd.concat(electrode_rows, ignore_index=True)
    electrode_path = OUT_DIR / "eeg_theta_per_electrode.csv"
    electrode_df.to_csv(electrode_path, index=False)
    print(f"Se escribió {electrode_path} ({len(electrode_df)} filas)")

if roi_rows:
    roi_df = pd.concat(roi_rows, ignore_index=True)
    roi_path = OUT_DIR / "eeg_theta_epochs_roi.csv"
    roi_df.to_csv(roi_path, index=False)
    print(f"Se escribió {roi_path} ({len(roi_df)} filas)")

if skipped:
    skipped_df = pd.DataFrame(skipped, columns=["participant_id", "session", "task", "reason"])
    skipped_path = OUT_DIR / "eeg_pipeline_skipped.csv"
    skipped_df.to_csv(skipped_path, index=False)
    print(f"Se escribió {skipped_path} ({len(skipped_df)} grabaciones descartadas)")
    skipped_df

Se escribió C:\Users\jimen\Downloads\data-visualization-project\deliveries\week06\data\processed\eeg_theta_per_electrode.csv (12993 filas)


Se escribió C:\Users\jimen\Downloads\data-visualization-project\deliveries\week06\data\processed\eeg_theta_epochs_roi.csv (47733 filas)
Se escribió C:\Users\jimen\Downloads\data-visualization-project\deliveries\week06\data\processed\eeg_pipeline_skipped.csv (5 grabaciones descartadas)
